# LeetCode 1309 — Decrypt String from Alphabet to Integer Mapping bằng Python

**Link đề:** https://leetcode.com/problems/decrypt-string-from-alphabet-to-integer-mapping/
**Độ khó:** Easy · **Chủ đề:** String (duyệt chuỗi bằng chỉ số, nhìn trước — *lookahead*)

```python
class Solution:
    def freqAlphabets(self, s: str) -> str:
        ...
```

## Mục tiêu học
1. Hiểu cách **giải mã** một chuỗi theo bảng quy tắc có hai loại "mã": 1 ký tự (`'1'`..`'9'`) và 3 ký tự (`'10#'`..`'26#'`).
2. Nắm công cụ Python: indexing, slicing `s[i:i+2]`, `int()`, `chr()`, `ord()`, `''.join()`, vòng `while` với chỉ số tự quản lý.
3. Học 4 lời giải: duyệt trái→phải nhìn trước 2 ô, duyệt phải→trái (theo hint), `replace` 26→10, và regex.
4. Trace từng bước, chứng minh tính đúng, Time/Space, lỗi người mới hay mắc.
5. Tự luyện: mã hoá ngược, đếm, biến thể.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho chuỗi `s` gồm chữ số và ký tự `'#'`. Ánh xạ sang chữ thường tiếng Anh như sau:

- `'a'` → `'i'` được mã hoá bằng `'1'` → `'9'`.
- `'j'` → `'z'` được mã hoá bằng `'10#'` → `'26#'`.

Trả về chuỗi sau khi giải mã. Đề đảm bảo luôn có **một cách giải mã duy nhất**.

| Input `s` | Output | Giải thích |
|---|---|---|
| `"10#11#12"` | `"jkab"` | `10#`→j, `11#`→k, `1`→a, `2`→b |
| `"1326#"` | `"acz"` | `1`→a, `3`→c, `26#`→z |

**Constraints:** `1 <= s.length <= 1000`, `s` chỉ gồm chữ số và `'#'`, luôn giải mã được.

Câu hỏi trước khi code:
- Đọc `"12"`: là `a` rồi `b`, hay là `l` (12)? Dựa vào đâu để quyết định?
- Ký tự `'#'` cho ta biết điều gì về **2 ký tự đứng ngay trước nó**?

## 2. Nền tảng Python

### 2.1. Indexing và slicing chuỗi

- `s[i]` lấy ký tự ở vị trí `i` (bắt đầu từ 0).
- `s[i:j]` lấy đoạn từ `i` đến **trước** `j`. Ví dụ `"1326#"[2:4]` → `"26"`.
- Truy cập `s[i]` với `i >= len(s)` sẽ **lỗi** `IndexError`, nhưng slicing vượt biên thì **không lỗi** (chỉ cắt ngắn lại).

In [1]:
s = "1326#"
print("s[0] =", s[0], "| s[4] =", s[4])
print("s[2:4] =", s[2:4])
print("slicing vượt biên s[3:10] =", repr(s[3:10]))
try:
    s[10]
except IndexError as e:
    print("s[10] -> IndexError:", e)

s[0] = 1 | s[4] = #
s[2:4] = 26
slicing vượt biên s[3:10] = '6#'
s[10] -> IndexError: string index out of range


### 2.2. `int()` đổi chuỗi số thành số nguyên

`int("26")` → `26`, `int("3")` → `3`. Ta sẽ dùng nó để biến mã `"26"` thành số 26.

In [2]:
print(int("26"), type(int("26")))
print(int("3") + 1)

26 <class 'int'>
4


### 2.3. `ord()` và `chr()` — ký tự ↔ mã số

Máy tính lưu mỗi ký tự bằng một số (mã Unicode).

- `ord('a')` → `97`, `ord('b')` → `98`, ..., `ord('z')` → `122`.
- `chr(97)` → `'a'` (ngược lại của `ord`).

Muốn đổi số thứ tự `k` (1..26) thành chữ cái: **`chr(ord('a') + k - 1)`**. Ví dụ `k = 1` → `'a'`, `k = 26` → `'z'`.

In [3]:
print(ord('a'), ord('b'), ord('z'))
print(chr(97), chr(122))
for k in [1, 2, 9, 10, 26]:
    print(k, "->", chr(ord('a') + k - 1))

97 98 122
a z
1 -> a
2 -> b
9 -> i
10 -> j
26 -> z


### 2.4. `''.join(list)` — ghép nhiều ký tự thành chuỗi

Chuỗi trong Python là **bất biến** (immutable): `res += ch` mỗi lần tạo chuỗi mới. Cách chuẩn là gom các ký tự vào `list` rồi `''.join(list)` một lần ở cuối.

In [4]:
parts = []
parts.append('j')
parts.append('k')
parts.append('a')
print(parts, "->", ''.join(parts))
print("-".join(["x", "y", "z"]))

['j', 'k', 'a'] -> jka
x-y-z


### 2.5. Vòng `while` với chỉ số tự quản lý

`for i in range(n)` luôn tăng `i` đúng 1. Bài này cần **nhảy 1 hoặc 3 bước** tuỳ trường hợp, nên dùng `while` và tự tăng `i`.

In [5]:
s = "abcdefg"
i = 0
while i < len(s):
    step = 3 if s[i] in "ad" else 1   # biểu thức điều kiện một dòng: A if cond else B
    print(f"i={i}, s[i]={s[i]}, nhảy {step}")
    i += step

i=0, s[i]=a, nhảy 3
i=3, s[i]=d, nhảy 3
i=6, s[i]=g, nhảy 1


Ở trên xuất hiện **biểu thức điều kiện** `A if cond else B`: nếu `cond` đúng lấy `A`, sai lấy `B`. Và toán tử `in` với chuỗi: `'a' in "ad"` → `True`.

## 3. Hiểu bài bằng tay — `'#'` là "dấu đóng mã 2 chữ số"

Đọc `"10#11#12"`:

```text
1 0 #  1 1 #  1  2
\___/  \___/  |  |
  j      k    a  b
```

Quan sát: **`'#'` luôn đứng sau đúng 2 chữ số**, và 2 chữ số đó tạo thành một mã `10..26`. Chữ số nào **không** bị `'#'` "nhận" thì là mã 1 ký tự `1..9`.

Vậy khi đứng ở vị trí `i`, ta chỉ cần hỏi: **`s[i + 2] == '#'`?**
- Có → mã là `s[i:i+2]`, nhảy `i += 3`.
- Không → mã là `s[i]`, nhảy `i += 1`.

In [6]:
s = "10#11#12"
for i in [0, 3, 6, 7]:
    has_hash = i + 2 < len(s) and s[i + 2] == '#'
    print(f"i={i}: s[i]={s[i]}, s[i+2]=={'#'!r}? {has_hash}")

i=0: s[i]=1, s[i+2]=='#'? True
i=3: s[i]=1, s[i+2]=='#'? True
i=6: s[i]=1, s[i+2]=='#'? False
i=7: s[i]=2, s[i+2]=='#'? False


## 4. Cách trực giác / baseline — tra bảng ánh xạ

Người mới thường nghĩ: "Lập bảng 26 dòng, mã → chữ". Ta có thể tạo `dict` từ mã sang chữ:

```python
table = {'1': 'a', ..., '9': 'i', '10#': 'j', ..., '26#': 'z'}
```

Rồi tại mỗi vị trí, **thử mã dài 3 trước**, nếu không có trong bảng thì thử mã dài 1. Đây là kiểu "tham lam theo độ dài" (greedy longest match).

`dict` là bảng tra cứu key → value, tra `key in table` nhanh trung bình `O(1)`.

In [7]:
table = {}
for k in range(1, 27):
    code = str(k) if k <= 9 else str(k) + '#'
    table[code] = chr(ord('a') + k - 1)
print("vài dòng của bảng:", {c: table[c] for c in ['1', '9', '10#', '26#']})
print("số mục:", len(table))

vài dòng của bảng: {'1': 'a', '9': 'i', '10#': 'j', '26#': 'z'}
số mục: 26


In [8]:
def decrypt_table(s: str) -> str:
    out = []
    i = 0
    while i < len(s):
        three = s[i:i + 3]          # slicing không lỗi kể cả gần cuối chuỗi
        if three in table:          # mã dài 3, ví dụ '10#'
            out.append(table[three])
            i += 3
        else:                       # mã dài 1
            out.append(table[s[i]])
            i += 1
    return ''.join(out)

print(decrypt_table("10#11#12"))
print(decrypt_table("1326#"))

jkab
acz


### 4.1. Baseline có vấn đề gì không?

Thử mã dài 3 trước là **đúng**: nếu `s[i:i+3]` có dạng `'dd#'` thì `'#'` ấy chắc chắn thuộc về 2 chữ số này (vì `'#'` luôn đóng đúng 2 chữ số ngay trước nó).

Nhưng cách này cần **xây bảng 26 mục** và tạo chuỗi con `s[i:i+3]` mỗi bước. Không chậm (vẫn `O(n)`), nhưng ta có thể làm gọn hơn: chỉ cần **nhìn 1 ký tự `s[i+2]`** và tính chữ bằng `chr`.

Phản ví dụ cho kiểu tham lam **sai** — thử mã dài **2 không có `#`** trước:

In [9]:
def wrong_greedy_two(s):
    out, i = [], 0
    while i < len(s):
        if i + 1 < len(s) and s[i + 1] != '#' and 10 <= int(s[i:i + 2]) <= 26:
            out.append(chr(ord('a') + int(s[i:i + 2]) - 1)); i += 2   # đọc "12" thành 'l' — sai luật đề
        elif i + 2 < len(s) and s[i + 2] == '#':
            out.append(chr(ord('a') + int(s[i:i + 2]) - 1)); i += 3
        else:
            out.append(chr(ord('a') + int(s[i]) - 1)); i += 1
    return ''.join(out)

for s in ["1112", "12", "1326"]:
    print(f"{s!r}: sai -> {wrong_greedy_two(s)!r} | đúng -> {decrypt_table(s)!r}")

'1112': sai -> 'kl' | đúng -> 'aaab'
'12': sai -> 'l' | đúng -> 'ab'
'1326': sai -> 'mz' | đúng -> 'acbf'


Lỗi: trong đề này, mã 2 chữ số **bắt buộc có `'#'`**. Không có `'#'` thì mỗi chữ số là một chữ cái riêng. Đó là lý do bài luôn có cách giải mã duy nhất.

## 5. Insight — chỉ cần nhìn trước 2 ô

**Quy tắc:** tại vị trí `i`, nếu `i + 2 < len(s)` **và** `s[i + 2] == '#'` thì 3 ký tự `s[i], s[i+1], '#'` là một mã; ngược lại `s[i]` là một mã.

Chữ cái được tính trực tiếp: `chr(ord('a') + int(code) - 1)` — không cần bảng.

Điều kiện `i + 2 < len(s)` phải đứng **trước** trong `and` để tránh `IndexError` (Python dừng đánh giá `and` ngay khi vế trái sai — gọi là *short-circuit*).

In [10]:
s = "12"
i = 1
print("i + 2 < len(s)?", i + 2 < len(s))
print("an toàn:", i + 2 < len(s) and s[i + 2] == '#')   # vế phải không được chạy

i + 2 < len(s)? False
an toàn: False


## 6. Lời giải chính — duyệt trái → phải, giảng từng đoạn

**Đoạn 1:** `out = []`, `i = 0`.

**Đoạn 2:** `while i < len(s)`:
- nếu `i + 2 < len(s) and s[i + 2] == '#'`: `num = int(s[i:i+2])`, `i += 3`.
- ngược lại: `num = int(s[i])`, `i += 1`.

**Đoạn 3:** `out.append(chr(ord('a') + num - 1))`.

**Đoạn 4:** `return ''.join(out)`.

In [11]:
def decrypt_left_to_right(s: str) -> str:
    out = []
    i = 0
    n = len(s)
    while i < n:
        if i + 2 < n and s[i + 2] == '#':
            num = int(s[i:i + 2])
            i += 3
        else:
            num = int(s[i])
            i += 1
        out.append(chr(ord('a') + num - 1))
    return ''.join(out)

for s in ["10#11#12", "1326#", "1", "26#", "12345678910#"]:
    print(repr(s), "->", decrypt_left_to_right(s))

'10#11#12' -> jkab
'1326#' -> acz
'1' -> a
'26#' -> z
'12345678910#' -> abcdefghij


## 7. TRACE từng bước

Bảng trace cho `s = "1326#"` (`len = 5`):

| i | s[i] | s[i+2] | có `#`? | mã | num | chữ | out sau bước | i mới |
|---|---|---|---|---|---|---|---|---|
| 0 | `1` | `2` | không | `1` | 1 | a | `[a]` | 1 |
| 1 | `3` | `6` | không | `3` | 3 | c | `[a,c]` | 2 |
| 2 | `2` | `#` | **có** | `26#` | 26 | z | `[a,c,z]` | 5 |

`i = 5 = len(s)` → dừng. Kết quả `"acz"`.

In [12]:
def trace_decrypt(s: str) -> str:
    out, i, n = [], 0, len(s)
    print(f"s = {s!r}, len = {n}")
    while i < n:
        look = s[i + 2] if i + 2 < n else None
        if i + 2 < n and s[i + 2] == '#':
            code, num, nxt = s[i:i + 3], int(s[i:i + 2]), i + 3
        else:
            code, num, nxt = s[i], int(s[i]), i + 1
        ch = chr(ord('a') + num - 1)
        out.append(ch)
        print(f"  i={i}: s[i+2]={look!r:5} mã={code!r:6} num={num:<3} -> {ch!r}  out={''.join(out)!r}  i mới={nxt}")
        i = nxt
    return ''.join(out)

trace_decrypt("1326#")
print()
trace_decrypt("10#11#12")

s = '1326#', len = 5
  i=0: s[i+2]='2'   mã='1'    num=1   -> 'a'  out='a'  i mới=1
  i=1: s[i+2]='6'   mã='3'    num=3   -> 'c'  out='ac'  i mới=2
  i=2: s[i+2]='#'   mã='26#'  num=26  -> 'z'  out='acz'  i mới=5

s = '10#11#12', len = 8
  i=0: s[i+2]='#'   mã='10#'  num=10  -> 'j'  out='j'  i mới=3
  i=3: s[i+2]='#'   mã='11#'  num=11  -> 'k'  out='jk'  i mới=6
  i=6: s[i+2]=None  mã='1'    num=1   -> 'a'  out='jka'  i mới=7
  i=7: s[i+2]=None  mã='2'    num=2   -> 'b'  out='jkab'  i mới=8


'jkab'

## 8. Code nộp LeetCode

In [13]:
class Solution:
    def freqAlphabets(self, s: str) -> str:
        out = []
        i, n = 0, len(s)
        while i < n:
            if i + 2 < n and s[i + 2] == '#':
                num = int(s[i:i + 2])
                i += 3
            else:
                num = int(s[i])
                i += 1
            out.append(chr(ord('a') + num - 1))
        return ''.join(out)

print(Solution().freqAlphabets("10#11#12"))
print(Solution().freqAlphabets("1326#"))

jkab
acz


## 9. Các lời giải khác

### Cách 3 — duyệt phải → trái (theo hint của đề)

Đi từ cuối chuỗi: nếu gặp `'#'` thì 2 ký tự **trước** nó là mã → lấy `s[i-2:i]`, nhảy `i -= 3`. Nếu là chữ số thì mã 1 ký tự, `i -= 1`. Vì đi ngược, cuối cùng phải **đảo** danh sách kết quả.

Ưu điểm: khi gặp `'#'` ta **biết chắc** đó là mã 2 chữ số, không cần kiểm tra biên `i + 2 < n`.

In [14]:
def decrypt_right_to_left(s: str) -> str:
    out = []
    i = len(s) - 1
    while i >= 0:
        if s[i] == '#':
            num = int(s[i - 2:i])
            i -= 3
        else:
            num = int(s[i])
            i -= 1
        out.append(chr(ord('a') + num - 1))
    return ''.join(reversed(out))     # reversed(...) duyệt ngược, không sửa list gốc

print(decrypt_right_to_left("10#11#12"))
print(decrypt_right_to_left("1326#"))
print("reversed demo:", list(reversed(['z', 'c', 'a'])))

jkab
acz
reversed demo: ['a', 'c', 'z']


### Cách 4 — thay thế từ lớn xuống nhỏ bằng `str.replace`

Thay `'26#'` → `'z'`, `'25#'` → `'y'`, ..., `'10#'` → `'j'`, rồi thay `'9'` → `'i'`, ..., `'1'` → `'a'`.

Thứ tự **phải** là mã có `#` trước, mã 1 chữ số sau. Nếu thay `'1'` → `'a'` trước thì `'10#'` bị phá thành `'a0#'`.

Độ phức tạp: 26 lần `replace`, mỗi lần `O(n)` → `O(26·n)` = `O(n)` nhưng hằng số lớn hơn.

In [15]:
def decrypt_replace(s: str) -> str:
    for k in range(26, 9, -1):                    # 26, 25, ..., 10
        s = s.replace(str(k) + '#', chr(ord('a') + k - 1))
    for k in range(9, 0, -1):                     # 9, ..., 1
        s = s.replace(str(k), chr(ord('a') + k - 1))
    return s

print(decrypt_replace("10#11#12"))
print(decrypt_replace("1326#"))
print("range(26, 9, -1) bắt đầu:", list(range(26, 9, -1))[:4], "...")

jkab
acz
range(26, 9, -1) bắt đầu: [26, 25, 24, 23] ...


### Cách 5 — biểu thức chính quy (regex)

Module chuẩn `re`: mẫu `r'\d\d#|\d'` nghĩa là "hai chữ số rồi `#`" **hoặc** "một chữ số". `re.findall` quét trái → phải, ưu tiên nhánh đầu (dài hơn) → đúng giống cách 2.

In [16]:
import re

def decrypt_regex(s: str) -> str:
    tokens = re.findall(r'\d\d#|\d', s)
    return ''.join(chr(ord('a') + int(t[:2]) - 1) for t in tokens)

print(re.findall(r'\d\d#|\d', "10#11#12"))
print(decrypt_regex("10#11#12"), decrypt_regex("1326#"))

['10#', '11#', '1', '2']
jkab acz


Ở đây `t[:2]` lấy tối đa 2 ký tự đầu: `'26#'[:2]` → `'26'`, `'3'[:2]` → `'3'`. Biểu thức `chr(...) for t in tokens` trong `join` là **generator expression** — giống list comprehension nhưng không tạo list trung gian.

### Code nộp LeetCode cho các cách còn lại

In [17]:
class SolutionRightToLeft:
    def freqAlphabets(self, s: str) -> str:
        out, i = [], len(s) - 1
        while i >= 0:
            if s[i] == '#':
                num = int(s[i - 2:i]); i -= 3
            else:
                num = int(s[i]); i -= 1
            out.append(chr(96 + num))     # 96 = ord('a') - 1
        return ''.join(reversed(out))

class SolutionReplace:
    def freqAlphabets(self, s: str) -> str:
        for k in range(26, 0, -1):
            s = s.replace(str(k) + ('#' if k >= 10 else ''), chr(96 + k))
        return s

class SolutionRegex:
    def freqAlphabets(self, s: str) -> str:
        return ''.join(chr(96 + int(t[:2])) for t in re.findall(r'\d\d#|\d', s))

for cls in [SolutionRightToLeft, SolutionReplace, SolutionRegex]:
    print(cls.__name__, cls().freqAlphabets("10#11#12"), cls().freqAlphabets("1326#"))

SolutionRightToLeft jkab acz
SolutionReplace jkab acz
SolutionRegex jkab acz


### Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Bảng dict | thử mã dài 3 rồi dài 1 | `O(n)` | `O(n)` | dễ hiểu, phải dựng bảng |
| 2. Trái→phải nhìn `s[i+2]` | lookahead 2 ô | `O(n)` | `O(n)` | lời giải chính, phải cẩn thận biên |
| 3. Phải→trái | gặp `#` thì lùi 3 | `O(n)` | `O(n)` | không lo biên, nhớ đảo kết quả |
| 4. `replace` | 26 lần thay, lớn trước | `O(26n)` | `O(n)` | ngắn, thứ tự rất quan trọng |
| 5. Regex | `\d\d#\|\d` | `O(n)` | `O(n)` | gọn, cần biết regex |

`n = len(s)`. Space `O(n)` là cho chuỗi kết quả.

## 10. Vì sao lời giải chính đúng?

**Invariant:** trước mỗi vòng `while`, `out` là bản giải mã chính xác của tiền tố `s[:i]`, và `i` luôn là **điểm bắt đầu của một mã**.

- Ban đầu `i = 0`, `out = []` — đúng.
- Tại `i`: nếu `s[i+2] == '#'` thì `'#'` này đóng đúng 2 chữ số `s[i], s[i+1]` (luật đề), nên mã là 3 ký tự. Nếu không thì `s[i+1]` không thể là phần của mã 2 chữ số bắt đầu ở `i` (mã đó cần `'#'` ở `i+2`), nên `s[i]` đứng riêng.
- Sau khi thêm chữ và nhảy đúng độ dài mã, `i` lại là điểm bắt đầu mã tiếp theo.

Khi `i == len(s)`, `out` là giải mã của cả chuỗi. ∎

### 10.1. Độ phức tạp

- **Time:** `O(n)` — mỗi ký tự của `s` được "tiêu thụ" đúng một lần (i chỉ tăng).
- **Space:** `O(n)` cho list `out` và chuỗi kết quả; bộ nhớ phụ khác `O(1)`.

## 11. Bộ test

Mẫu đề, biên (1 ký tự, chỉ có 1 mã `#`), mọi mã `1..26`, và **kiểm thử đối chiếu**: tự sinh chuỗi ngẫu nhiên bằng cách **mã hoá** một chuỗi chữ thường, rồi giải mã phải ra lại chuỗi ban đầu.

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS -> {actual!r}")

sol = Solution()
check("ví dụ 1", sol.freqAlphabets("10#11#12"), "jkab")
check("ví dụ 2", sol.freqAlphabets("1326#"), "acz")
check("1 ký tự", sol.freqAlphabets("1"), "a")
check("chỉ một mã #", sol.freqAlphabets("26#"), "z")
check("9 chữ đầu", sol.freqAlphabets("123456789"), "abcdefghi")
check("10# liền 1", sol.freqAlphabets("10#1"), "ja")
check("1 rồi 10#", sol.freqAlphabets("110#"), "aj")

ví dụ 1                PASS -> 'jkab'
ví dụ 2                PASS -> 'acz'
1 ký tự                PASS -> 'a'
chỉ một mã #           PASS -> 'z'
9 chữ đầu              PASS -> 'abcdefghi'
10# liền 1             PASS -> 'ja'
1 rồi 10#              PASS -> 'aj'


In [19]:
import random

def encode(word: str) -> str:
    parts = []
    for ch in word:
        k = ord(ch) - ord('a') + 1
        parts.append(str(k) if k <= 9 else str(k) + '#')
    return ''.join(parts)

print("encode('jkab') =", encode("jkab"))

rng = random.Random(1309)
methods = [decrypt_table, decrypt_left_to_right, decrypt_right_to_left, decrypt_replace, decrypt_regex,
           Solution().freqAlphabets]
for _ in range(500):
    word = ''.join(rng.choice('abcdefghijklmnopqrstuvwxyz') for _ in range(rng.randint(1, 40)))
    s = encode(word)
    for f in methods:
        assert f(s) == word, (f, s, word)
print("500 chuỗi ngẫu nhiên: mọi cách giải mã đều ra đúng chuỗi gốc")

encode('jkab') = 10#11#12
500 chuỗi ngẫu nhiên: mọi cách giải mã đều ra đúng chuỗi gốc


## 12. Lỗi người mới hay gặp

### Lỗi 1: Kiểm tra `s[i + 2]` mà quên biên → `IndexError`

```python
if s[i + 2] == '#':      # i gần cuối thì vượt biên!
```

**Sửa:** `if i + 2 < len(s) and s[i + 2] == '#'` (vế kiểm tra biên đứng trước).

In [20]:
def wrong_no_bound(s):
    out, i = [], 0
    while i < len(s):
        if s[i + 2] == '#':
            out.append(chr(96 + int(s[i:i + 2]))); i += 3
        else:
            out.append(chr(96 + int(s[i]))); i += 1
    return ''.join(out)

print("'1326#' ->", wrong_no_bound("1326#"), "(may mắn: chuỗi kết thúc bằng mã có '#')")
try:
    print(wrong_no_bound("10#12"))
except IndexError as e:
    print("'10#12' -> IndexError:", e)

'1326#' -> acz (may mắn: chuỗi kết thúc bằng mã có '#')
'10#12' -> IndexError: string index out of range


### Lỗi 2: Kiểm tra `s[i + 1] == '#'` (lệch 1 ô)

`'#'` nằm sau **2** chữ số, tức ở `i + 2`, không phải `i + 1`.

### Lỗi 3: Lệch 1 khi đổi số sang chữ

`chr(ord('a') + num)` cho `1 → 'b'`. Phải trừ 1: `chr(ord('a') + num - 1)`.

### Lỗi 4: Cách `replace` nhưng thay mã 1 chữ số trước

In [21]:
print("lệch 1  :", chr(ord('a') + 1), "| đúng:", chr(ord('a') + 1 - 1))

def wrong_replace_order(s):
    for k in range(1, 27):                 # thay '1' trước -> phá '10#'
        s = s.replace(str(k) + ('#' if k >= 10 else ''), chr(96 + k))
    return s

print("replace sai thứ tự:", wrong_replace_order("10#11#12"), "| đúng:", decrypt_replace("10#11#12"))

lệch 1  : b | đúng: a
replace sai thứ tự: a0#aa#ab | đúng: jkab


### Lỗi 5: Dùng `res += ch` trong vòng lặp lớn

Vẫn chạy đúng (CPython thường tối ưu được), nhưng thói quen tốt là gom vào `list` rồi `''.join` — chi phí `O(n)` được đảm bảo.

## 13. Mẹo nhận dạng pattern

- Chuỗi có **ký hiệu đánh dấu** (`#`, `*`, dấu ngoặc...) quyết định cách đọc các ký tự lân cận ⇒ **duyệt bằng chỉ số + nhìn trước/nhìn sau** (lookahead / lookbehind).
- Nếu ký hiệu đứng **sau** phần nó điều khiển (như `#` ở đây) ⇒ duyệt **phải → trái** thường gọn hơn.
- Bước nhảy không cố định ⇒ dùng `while` thay vì `for ... in range`.
- Đổi số thứ tự ↔ chữ cái: `chr(ord('a') + k - 1)` và `ord(ch) - ord('a') + 1`.

## 14. Biến thể

1. **Mã hoá ngược** (encode): cho chuỗi chữ thường, sinh chuỗi mã (đã viết ở phần test).
2. **Mã có `#` ở đầu** (ví dụ `#10` thay vì `10#`): khi đó duyệt trái → phải, gặp `#` thì đọc 2 ký tự sau.
3. **LeetCode 91 — Decode Ways**: không có `#`, đếm số cách giải mã ⇒ quy hoạch động (khó hơn nhiều vì mập mờ).

In [22]:
def decrypt_prefix_hash(s: str) -> str:
    out, i = [], 0
    while i < len(s):
        if s[i] == '#':
            out.append(chr(96 + int(s[i + 1:i + 3]))); i += 3
        else:
            out.append(chr(96 + int(s[i]))); i += 1
    return ''.join(out)

print(decrypt_prefix_hash("#10#1112"))   # j k a b

jkab


## 15. Bài tập tự luyện (TODO — tăng dần)

**Bài 1 (tính tay):** Giải mã `"25#1324#9"`. Lập bảng trace `i`, mã, chữ.

**Bài 2 (đoán output):** `chr(ord('a') + 12 - 1)` in ra gì? `ord('z') - ord('a') + 1` bằng bao nhiêu?

**Bài 3 (điền chỗ trống):** hoàn thiện `exercise_decrypt` theo cách duyệt phải → trái.

**Bài 4 (tự viết):** `exercise_encode(word)` — mã hoá chuỗi chữ thường thành chuỗi mã theo đề.

**Bài 5 (biến thể):** `exercise_count_hash_letters(s)` — đếm bao nhiêu chữ trong kết quả thuộc nhóm `'j'..'z'`, **không** tạo chuỗi kết quả.

**Bài 6 (mini challenge):** `exercise_decrypt_upper(s)` — như đề nhưng mã `27#..52#` biểu diễn `'A'..'Z'`.

In [23]:
def exercise_decrypt(s: str) -> str:
    out = []
    i = len(s) - 1
    # TODO: while i >= 0 ... nếu s[i] == '#' lấy 2 ký tự trước, lùi 3; ngược lại lùi 1
    # TODO: nhớ đảo kết quả
    pass

# assert exercise_decrypt("10#11#12") == "jkab"

In [24]:
def exercise_encode(word: str) -> str:
    # TODO
    pass

def exercise_count_hash_letters(s: str) -> int:
    # TODO: gợi ý — mỗi chữ thuộc j..z ứng với đúng một ký tự '#'
    pass

# assert exercise_encode("acz") == "1326#"
# assert exercise_count_hash_letters("10#11#12") == 2

In [25]:
def exercise_decrypt_upper(s: str) -> str:
    # TODO: 1..9 -> a..i ; 10#..26# -> j..z ; 27#..52# -> A..Z
    pass

# assert exercise_decrypt_upper("127#52#26#") == "aAZz"

## 16. Gợi ý 3 mức

**Bài 1** — (1) Tìm các `#` trước. (2) Mỗi `#` đi với 2 chữ số ngay trước. (3) `25#`, `1`, `3`, `24#`, `9`.

**Bài 3** — (1) Đi từ `len(s)-1` về 0. (2) `s[i-2:i]` là 2 chữ số trước `#`. (3) `''.join(reversed(out))`.

**Bài 4** — (1) `k = ord(ch) - ord('a') + 1`. (2) `k <= 9` thì `str(k)`, ngược lại `str(k) + '#'`. (3) Gom vào list rồi `join`.

**Bài 5** — (1) Có cần giải mã không? (2) Mỗi chữ `j..z` sinh ra đúng một `'#'`. (3) `s.count('#')`.

**Bài 6** — (1) Logic tách mã giữ nguyên. (2) Chỉ đổi phần `num → chữ`. (3) `num <= 26` → `chr(96 + num)`, ngược lại `chr(ord('A') + num - 27)`.

## 17. Đáp án tham khảo

**Bài 1:** `25#`→y, `1`→a, `3`→c, `24#`→x, `9`→i ⇒ `"yacxi"`.

**Bài 2:** `chr(ord('a') + 11)` = `'l'`; `ord('z') - ord('a') + 1` = `26`.

In [26]:
def answer_decrypt(s):
    out, i = [], len(s) - 1
    while i >= 0:
        if s[i] == '#':
            out.append(chr(96 + int(s[i - 2:i]))); i -= 3
        else:
            out.append(chr(96 + int(s[i]))); i -= 1
    return ''.join(reversed(out))

def answer_encode(word):
    parts = []
    for ch in word:
        k = ord(ch) - ord('a') + 1
        parts.append(str(k) if k <= 9 else f"{k}#")
    return ''.join(parts)

def answer_count_hash_letters(s):
    return s.count('#')

def answer_decrypt_upper(s):
    out, i = [], 0
    while i < len(s):
        if i + 2 < len(s) and s[i + 2] == '#':
            num = int(s[i:i + 2]); i += 3
        else:
            num = int(s[i]); i += 1
        out.append(chr(96 + num) if num <= 26 else chr(ord('A') + num - 27))
    return ''.join(out)

assert answer_decrypt("25#1324#9") == "yacxi"
assert chr(ord('a') + 12 - 1) == 'l' and ord('z') - ord('a') + 1 == 26
assert answer_decrypt("10#11#12") == "jkab"
assert answer_encode("acz") == "1326#"
assert answer_count_hash_letters("10#11#12") == 2
assert answer_decrypt_upper("127#52#26#") == "aAZz"
for w in ["abc", "xyz", "hello", "z", "jjj"]:
    assert answer_decrypt(answer_encode(w)) == w
    assert answer_count_hash_letters(answer_encode(w)) == sum(ch >= 'j' for ch in w)
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def freqAlphabets(self, s: str) -> str:
        out, i, n = [], 0, len(s)
        while i < n:
            if i + 2 < n and s[i + 2] == '#':   # mã 2 chữ số + '#'
                num = int(s[i:i + 2]); i += 3
            else:                                # mã 1 chữ số
                num = int(s[i]); i += 1
            out.append(chr(ord('a') + num - 1))
        return ''.join(out)
```

**Checklist:**
- [ ] Kiểm tra biên `i + 2 < n` **trước** khi đọc `s[i + 2]`.
- [ ] `'#'` ở vị trí `i + 2`, không phải `i + 1`.
- [ ] Đổi số → chữ có `- 1`.
- [ ] Nhảy `3` khi có `#`, nhảy `1` khi không.
- [ ] Đã test chuỗi kết thúc bằng mã 1 chữ số và bằng mã `#`.